In [ ]:
"""
Method 1: Unstructured Pruning — ResNet-50 / CIFAR-10
======================================================
Global L1 magnitude pruning — removes individual weights (fine-grained).
Weights become zero but tensors stay dense (no structural change).

Saves exactly 8 metrics per sparsity level for comparison with baseline:
  accuracy, precision, recall, f1, num_parameters, model_size_mb,
  inference_time_ms, flops

Output: method1_unstructured_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method1_unstructured_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS = [0.30, 0.50, 0.70, 0.80, 0.90]
INFERENCE_RUNS  = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning ───────────────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def apply_pruning(model, sparsity):
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, param_name in layers:
        prune.remove(module, param_name)
    return pruned


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    """Total number of parameters (including zeroed-out pruned weights)."""
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    """Actual disk size of saved .pth file in MB."""
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    """
    Average single-batch inference time in ms.
    Uses GPU if available, otherwise CPU.
    """
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        # GPU timing with CUDA events
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        # CPU timing
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    """FLOPs via THOP (MACs × 2). Shape-based — same for all sparsity levels."""
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Method 1: Unstructured Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device : {DEVICE}")
    print(f"  Sparsity levels: {[int(s*100) for s in SPARSITY_LEVELS]}%")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    results = []

    for sparsity in SPARSITY_LEVELS:
        print(f"  Pruning at {int(sparsity*100)}% sparsity...")
        pruned = apply_pruning(model, sparsity)

        metrics       = evaluate(pruned, loader)
        num_params    = count_parameters(pruned)
        model_size_mb = get_model_size_mb(pruned)
        inference_ms  = measure_inference_time_ms(pruned)
        flops         = compute_flops(pruned)

        row = {
            "sparsity"         : sparsity,
            # ── 8 standardized metrics ──
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "num_parameters"   : num_params,
            "model_size_mb"    : round(model_size_mb, 4),
            "inference_time_ms": round(inference_ms, 4),
            "flops"            : flops,
        }
        results.append(row)

        print(f"    Acc: {metrics['accuracy']:.4f} | Params: {num_params:,} | "
              f"Size: {model_size_mb:.2f} MB | Inf: {inference_ms:.2f} ms | "
              f"FLOPs: {flops/1e9:.3f} G")

    report = {
        "method"     : "unstructured_pruning",
        "description": "Global L1 unstructured pruning — individual weights zeroed globally",
        "baseline"   : baseline,
        "results"    : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Method 1: Unstructured Pruning — ResNet-50 / CIFAR-10
  Device : cuda
  Sparsity levels: [30, 50, 70, 80, 90]%

  Pruning at 30% sparsity...
    Acc: 0.9321 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.77 ms | FLOPs: 2.623 G
  Pruning at 50% sparsity...
    Acc: 0.9320 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.03 ms | FLOPs: 2.623 G
  Pruning at 70% sparsity...
    Acc: 0.9319 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.60 ms | FLOPs: 2.623 G
  Pruning at 80% sparsity...
    Acc: 0.9276 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.00 ms | FLOPs: 2.623 G
  Pruning at 90% sparsity...
    Acc: 0.8774 | Params: 23,520,842 | Size: 90.03 MB | Inf: 50.92 ms | FLOPs: 2.623 G

  ✓ Saved → method1_unstructured_pruning.json
